# TRAMA Visual Factory — Kaggle Q4 Smoke Test v0.1

**One-cell execution.** Enable a free GPU in Kaggle Settings, keep Internet ON, then run the single code cell.

This is a zero-cost technical qualification only. It does not publish or authorize student runtime.

In [ ]:
import os, sys, json, time, shutil, subprocess, hashlib, traceback
from pathlib import Path

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

receipt = {
    "schemaVersion": "trama.visual-factory.kaggle-q4-smoke/v0.1",
    "candidateId": "MZ-VF-001",
    "adapter": "kaggle-notebooks-free",
    "paidComputeAuthorized": False,
    "decision": "UNKNOWN",
}

# 1) Hardware qualification
if shutil.which("nvidia-smi") is None:
    receipt["decision"] = "KAGGLE_NO_GPU"
    print(json.dumps(receipt, indent=2))
    raise SystemExit("STOP: enable Accelerator -> GPU in Kaggle Settings and rerun.")

gpu_names = [x.strip() for x in sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines() if x.strip()]
vram_values = [float(x.strip())/1024 for x in sh("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits").splitlines() if x.strip()]
ram_kib = int(sh("awk '/MemTotal/ {print $2}' /proc/meminfo"))
ram_gib = ram_kib / 1024 / 1024

# Use one T4 only; multi-GPU is not part of the v0.1 proof.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

candidates = [Path("/kaggle/temp"), Path("/tmp"), Path("/kaggle/working")]
scratch = None
for p in candidates:
    try:
        p.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage(str(p)).free / (1024**3)
        if free >= 35:
            scratch = p / "trama-vf-q4"
            scratch.mkdir(parents=True, exist_ok=True)
            break
    except Exception:
        pass

if scratch is None:
    receipt.update({
        "gpuNames": gpu_names,
        "vramGiB": [round(v,2) for v in vram_values],
        "ramGiB": round(ram_gib,2),
        "decision": "KAGGLE_INSUFFICIENT_DISK",
    })
    print(json.dumps(receipt, indent=2))
    raise SystemExit("STOP: no scratch volume with at least 35 GiB free.")

receipt.update({
    "gpuNames": gpu_names,
    "vramGiB": [round(v,2) for v in vram_values],
    "ramGiB": round(ram_gib,2),
    "scratch": str(scratch),
    "scratchFreeGiB": round(shutil.disk_usage(str(scratch)).free/(1024**3),2),
})

if max(vram_values) < 14 or ram_gib < 20:
    receipt["decision"] = "KAGGLE_HARDWARE_BELOW_Q4_ENVELOPE"
    print(json.dumps(receipt, indent=2))
    raise SystemExit("STOP: allocated hardware is below the Q4 smoke-test envelope.")

print("Hardware accepted:", json.dumps(receipt, indent=2))

# 2) Install ComfyUI + GGUF
COMFY = scratch / "ComfyUI"
if not COMFY.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",str(COMFY)], check=True)

subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(COMFY/"requirements.txt")], check=True)

GGUF = COMFY/"custom_nodes"/"ComfyUI-GGUF"
if not GGUF.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/city96/ComfyUI-GGUF.git",str(GGUF)], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(GGUF/"requirements.txt")], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","huggingface_hub>=0.27","requests","pillow"], check=True)

# 3) Download the bounded Q4 package
from huggingface_hub import hf_hub_download
targets = {
    "unet": COMFY/"models"/"unet",
    "text": COMFY/"models"/"text_encoders",
    "vae": COMFY/"models"/"vae",
    "lora": COMFY/"models"/"loras",
}
for p in targets.values():
    p.mkdir(parents=True, exist_ok=True)

specs = [
    ("Aitrepreneur/FLX","Qwen_Image_Edit-Q4_K_S.gguf",targets["unet"]),
    ("Aitrepreneur/FLX","Qwen2.5-VL-7B-Instruct-UD-Q4_K_S.gguf",targets["text"]),
    ("Aitrepreneur/FLX","Qwen2.5-VL-7B-Instruct-mmproj-BF16.gguf",targets["text"]),
    ("Aitrepreneur/FLX","qwen_image_vae.safetensors",targets["vae"]),
    ("Aitrepreneur/FLX","Qwen-Image-Lightning-4steps-V1.0.safetensors",targets["lora"]),
]

downloaded = []
for repo_id, filename, folder in specs:
    path_local = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(folder))
    downloaded.append(path_local)
    print("ready:", filename)

def sha256(path):
    h = hashlib.sha256()
    with open(path,"rb") as fh:
        for chunk in iter(lambda: fh.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

receipt["modelFiles"] = {
    os.path.basename(p): {"bytes": os.path.getsize(p), "sha256": sha256(p)}
    for p in downloaded
}

# 4) Create neutral input plate
from PIL import Image, ImageDraw
img = Image.new("RGB",(512,512),(18,25,36))
d = ImageDraw.Draw(img)
d.rectangle((40,70,472,450), fill=(34,45,60), outline=(120,140,160), width=3)
d.rectangle((160,130,390,300), fill=(25,70,90), outline=(80,190,210), width=4)
d.rectangle((65,340,445,370), fill=(70,85,100))
d.ellipse((75,310,105,340), fill=(230,150,40))
input_dir = COMFY/"input"
input_dir.mkdir(parents=True, exist_ok=True)
img.save(input_dir/"trama_q4_smoke.png")

# 5) Start ComfyUI
import requests
log_path = scratch/"comfyui-q4-smoke.log"
logf = open(log_path,"w")
proc = subprocess.Popen(
    [sys.executable,"main.py","--lowvram","--reserve-vram","0.8","--listen","127.0.0.1","--port","8188","--disable-api-nodes"],
    cwd=str(COMFY),
    stdout=logf,
    stderr=subprocess.STDOUT,
)
base = "http://127.0.0.1:8188"
object_info = None
for _ in range(120):
    try:
        r = requests.get(base+"/object_info",timeout=2)
        if r.ok:
            object_info = r.json()
            break
    except Exception:
        pass
    time.sleep(2)

if object_info is None:
    logf.flush()
    receipt["decision"] = "Q4_SMOKE_FAIL_STARTUP"
    receipt["logTail"] = open(log_path,errors="ignore").read()[-6000:]
    print(json.dumps(receipt, indent=2))
    raise SystemExit("ComfyUI failed to start.")

required = ["LoadImage","VAELoader","UnetLoaderGGUF","TextEncodeQwenImageEditPlus","VAEEncode","KSampler","VAEDecode","SaveImage","CLIPLoaderGGUF","LoraLoaderModelOnly"]
missing = [n for n in required if n not in object_info]
if missing:
    receipt["decision"] = "Q4_SMOKE_FAIL_NODES"
    receipt["missingNodes"] = missing
    print(json.dumps(receipt, indent=2))
    raise SystemExit(f"Required nodes missing: {missing}")

# 6) Build one self-contained workflow
clip_inputs = {"clip_name":"Qwen2.5-VL-7B-Instruct-UD-Q4_K_S.gguf","type":"qwen_image"}
clip_required = object_info["CLIPLoaderGGUF"]["input"]["required"]
if "device" in clip_required:
    clip_inputs["device"] = "default"

def enc(prompt):
    x = {"clip":["2",0],"prompt":prompt,"vae":["3",0]}
    req = object_info["TextEncodeQwenImageEditPlus"]["input"]["required"]
    for key in ("image1","image2","image3"):
        if key in req:
            x[key] = ["1",0]
    return x

workflow = {
    "1":{"class_type":"LoadImage","inputs":{"image":"trama_q4_smoke.png"}},
    "2":{"class_type":"CLIPLoaderGGUF","inputs":clip_inputs},
    "3":{"class_type":"VAELoader","inputs":{"vae_name":"qwen_image_vae.safetensors"}},
    "4":{"class_type":"UnetLoaderGGUF","inputs":{"unet_name":"Qwen_Image_Edit-Q4_K_S.gguf"}},
    "13":{"class_type":"LoraLoaderModelOnly","inputs":{"model":["4",0],"lora_name":"Qwen-Image-Lightning-4steps-V1.0.safetensors","strength_model":1.0}},
    "5":{"class_type":"TextEncodeQwenImageEditPlus","inputs":enc("Transform this blockout into a polished contemporary museum interior after closing, cinematic editorial illustration, restrained warm amber and cool blue lighting, clean architecture, realistic exhibition materials, no text, no labels, no watermark, no people.")},
    "6":{"class_type":"TextEncodeQwenImageEditPlus","inputs":enc("text, labels, watermark, poster, infographic, dashboard, people, distorted geometry")},
    "7":{"class_type":"VAEEncode","inputs":{"pixels":["1",0],"vae":["3",0]}},
    "8":{"class_type":"KSampler","inputs":{"model":["13",0],"seed":20261004,"steps":4,"cfg":1.0,"sampler_name":"euler","scheduler":"simple","positive":["5",0],"negative":["6",0],"latent_image":["7",0],"denoise":1.0}},
    "11":{"class_type":"VAEDecode","inputs":{"samples":["8",0],"vae":["3",0]}},
    "12":{"class_type":"SaveImage","inputs":{"images":["11",0],"filename_prefix":"TRAMA_KAGGLE_Q4_SMOKE"}},
}

# 7) Run once, fail closed
try:
    q = requests.post(base+"/prompt",json={"prompt":workflow},timeout=30)
    q.raise_for_status()
    prompt_id = q.json()["prompt_id"]
    result = None
    for _ in range(900):
        h = requests.get(base+f"/history/{prompt_id}",timeout=10).json()
        if prompt_id in h:
            result = h[prompt_id]
            if result.get("outputs"):
                break
        time.sleep(2)

    if not result or not result.get("outputs"):
        raise RuntimeError("No output returned before timeout/completion.")

    images = result["outputs"].get("12",{}).get("images",[])
    if not images:
        raise RuntimeError("SaveImage produced no image metadata.")

    meta = images[0]
    out_path = COMFY/"output"/meta.get("subfolder","")/meta["filename"]
    final_copy = Path("/kaggle/working")/"TRAMA_KAGGLE_Q4_SMOKE.png"
    shutil.copy2(out_path, final_copy)

    receipt.update({
        "decision":"Q4_SMOKE_PASS",
        "promptId":prompt_id,
        "outputFile":str(final_copy),
    })
except Exception as e:
    logf.flush()
    tail = open(log_path,errors="ignore").read()[-12000:]
    low = (str(e)+"\n"+tail).lower()
    if "out of memory" in low or "cuda oom" in low:
        receipt["decision"] = "Q4_OOM_FALLBACK_Q3_ALLOWED"
    else:
        receipt["decision"] = "Q4_SMOKE_FAIL_DIAGNOSE"
    receipt["error"] = str(e)
    receipt["logTail"] = tail[-6000:]

with open("/kaggle/working/trama-kaggle-q4-smoke-receipt.json","w",encoding="utf-8") as fh:
    json.dump(receipt,fh,indent=2)

print(json.dumps(receipt,indent=2))
print("Receipt saved to /kaggle/working/trama-kaggle-q4-smoke-receipt.json")
if receipt["decision"] == "Q4_SMOKE_PASS":
    print("Image saved to /kaggle/working/TRAMA_KAGGLE_Q4_SMOKE.png")
